<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.8-conversational-rag/notebooks/exercises-6_8.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.8: Conversational RAG

*Module 6 · Retrieval-Augmented Generation (RAG)*

> Users don’t ask one question and leave. They have conversations: “What’s the refund policy?” then “How long does IT take?” then “Can I do THAT by email?”. The pronouns “it” and “that” only make sense with conversation history. This lesson builds a conversational RAG that reformulates queries, manages memory, and handles follow-ups — the system behind every production chatbot.

`Query Reformulation` · `Chat Memory` · `Follow-Up Handling` · `Production` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.8.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Query Reformulation — The Missing Piece — `01_reformulation.py`
2. Step 2 — Full Conversational RAG — Memory + Retrieval + Generation — `02_conversational_rag.py`
3. Step 3 — Memory Management — Sliding Window + Summarization — `03_memory.py`
4. Step 4 — Production Chatbot — Everything Together — `04_production_chatbot.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Query Reformulation — The Missing Piece

Use the LLM to rewrite follow-ups into standalone queries before retrieval.

**`01_reformulation.py`** · _Block 1 of 4_

QUERY REFORMULATION — Resolve pronouns + context


In [ ]:
# QUERY REFORMULATION — Resolve pronouns + context
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class QueryReformulator:
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def reformulate(self, follow_up, history):
        if not history: return follow_up
        hist_text = "\n".join(f"{m['role']}: {m['content']}" for m in history[-6:])
        resp = self.model.generate_content(
            f"Rewrite this follow-up as a STANDALONE question using the conversation history.\n"
            f"Resolve all pronouns (it, that, they, this, etc.).\n"
            f"Return ONLY the rewritten question.\n\n"
            f"History:\n{hist_text}\n\nFollow-up: {follow_up}\n\nStandalone question:"
        )
        return resp.text.strip()

# Demo
reformulator = QueryReformulator()

history = [
    {"role": "user", "content": "What courses does Netsetos offer?"},
    {"role": "assistant", "content": "Netsetos offers GenAI, Data Science, Cloud, and Full-Stack courses."},
]

follow_ups = [
    "How much does it cost?",         # it = GenAI course (most relevant)
    "Can I get a refund on that?",     # that = the course
    "What are the prerequisites?",     # already standalone
    "Is there an EMI option for it?",  # it = the course
]

print("Query Reformulation:\n")
for f in follow_ups:
    standalone = reformulator.reformulate(f, history)
    print(f"  Follow-up:   '{f}'")
    print(f"  Standalone:  '{standalone}'\n")


> **What just happened?** “How much does it cost?” was reformulated to “How much does the Netsetos GenAI course cost?” by resolving “it” from conversation history. This standalone query can now be embedded and searched against the vector store. Without reformulation, the retriever searches for “it” and finds nothing useful.


### Step 2 · Full Conversational RAG — Memory + Retrieval + Generation

Complete pipeline: reformulate → retrieve → generate → save to memory.

**`02_conversational_rag.py`** · _Block 2 of 4_

CONVERSATIONAL RAG — Complete pipeline


In [ ]:
# CONVERSATIONAL RAG — Complete pipeline
import google.generativeai as genai
import numpy as np, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class ConversationalRAG:
    def __init__(self, docs, max_history=10):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.history = []
        self.max_history = max_history
        # Build vector index
        self.chunks, self.embs = [], []
        for d in docs:
            emb = genai.embed_content(model="models/text-embedding-004", content=d["text"])["embedding"]
            self.chunks.append(d); self.embs.append(np.array(emb))

    def _reformulate(self, query):
        if not self.history: return query
        hist = "\n".join(f"{m['role']}: {m['content'][:100]}" for m in self.history[-6:])
        r = self.model.generate_content(
            f"Rewrite as standalone question. Resolve pronouns.\nHistory:\n{hist}\nFollow-up: {query}\nStandalone:")
        return r.text.strip()

    def _retrieve(self, query, k=3):
        qe = np.array(genai.embed_content(model="models/text-embedding-004", content=query)["embedding"])
        scores = [(i, np.dot(qe,e)/(np.linalg.norm(qe)*np.linalg.norm(e))) for i,e in enumerate(self.embs)]
        scores.sort(key=lambda x:-x[1])
        return [self.chunks[i] for i,_ in scores[:k]]

    def chat(self, user_msg):
        # Step 1: Reformulate with history
        standalone = self._reformulate(user_msg)

        # Step 2: Retrieve with standalone query
        docs = self._retrieve(standalone)
        context = "\n".join(d["text"] for d in docs)

        # Step 3: Generate with context + history
        hist_text = "\n".join(f"{m['role']}: {m['content'][:80]}" for m in self.history[-4:])
        resp = self.model.generate_content(
            f"Answer from context ONLY. Be conversational.\n\nContext:\n{context}\n\n"
            f"Conversation:\n{hist_text}\nUser: {user_msg}\nAssistant:")
        answer = resp.text.strip()

        # Step 4: Save to memory
        self.history.append({"role": "user", "content": user_msg})
        self.history.append({"role": "assistant", "content": answer})
        if len(self.history) > self.max_history * 2:
            self.history = self.history[-self.max_history * 2:]

        return {"answer": answer, "standalone_query": standalone,
                "sources": [d.get("source","?") for d in docs]}

# ── Build and test ──
docs = [
    {"text": "Netsetos offers GenAI, Data Science, Cloud, and Full-Stack courses.", "source": "about"},
    {"text": "GenAI course costs 14999 rupees. Lifetime access, Discord, weekly live sessions, certificate.", "source": "pricing"},
    {"text": "Refund policy: full refund 7 days, 50% 8-30 days, none after 30. Processed in 5 business days.", "source": "refund"},
    {"text": "Prerequisites: basic Python and high school math. No ML experience needed.", "source": "prereqs"},
    {"text": "Live classes daily 7 PM IST. Recordings in 2 hours. EMI available via Razorpay.", "source": "schedule"},
]

bot = ConversationalRAG(docs)
conversation = [
    "What courses does Netsetos offer?",
    "How much does the GenAI one cost?",   # "the GenAI one"
    "Can I pay in installments?",         # "pay" = for the course
    "What if I don't like it?",           # "it" = the course
    "How long do I have for that?",       # "that" = refund
]

print("Conversational RAG:\n")
for msg in conversation:
    r = bot.chat(msg)
    print(f"  You:  {msg}")
    print(f"  Bot:  {r['answer'][:120]}")
    print(f"  [reformulated: '{r['standalone_query'][:60]}' | sources: {r['sources']}]\n")


> **What just happened?** 5-turn conversation with progressively vague follow-ups. “How much does the GenAI one cost?” reformulated to include “Netsetos GenAI course.” “What if I don’t like it?” reformulated to “What is the refund policy for the GenAI course?” Each standalone query retrieved the correct document. Without reformulation, turns 2-5 would all fail retrieval because of pronouns and implicit references.


### Step 3 · Memory Management — Sliding Window + Summarization

Conversations can be 50+ turns. You can't send all of them. Two strategies.

**`03_memory.py`** · _Block 3 of 4_

MEMORY MANAGEMENT — Sliding window + summarization


In [ ]:
# MEMORY MANAGEMENT — Sliding window + summarization
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class ChatMemory:
    """Two strategies: sliding window (fast) and summarize (smart)."""

    def __init__(self, strategy="window", window_size=6):
        self.strategy = strategy
        self.window_size = window_size
        self.messages = []
        self.summary = ""
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})

    def get_context(self):
        if self.strategy == "window":
            # Keep last N messages
            return self.messages[-self.window_size:]
        elif self.strategy == "summarize":
            # Summarize old messages, keep recent
            if len(self.messages) > self.window_size:
                old = self.messages[:-self.window_size]
                old_text = "\n".join(f"{m['role']}: {m['content'][:80]}" for m in old)
                resp = self.model.generate_content(
                    f"Summarize this conversation in 2-3 sentences:\n{old_text}")
                self.summary = resp.text.strip()
                self.messages = self.messages[-self.window_size:]
            context = self.messages[:]
            if self.summary:
                context.insert(0, {"role":"system", "content":f"Previous: {self.summary}"})
            return context

# Demo
print("Memory Strategies:\n")
for strat in ["window", "summarize"]:
    mem = ChatMemory(strategy=strat, window_size=4)
    for i in range(8):
        mem.add("user", f"Question {i+1} about the course")
        mem.add("assistant", f"Answer {i+1} with details")
    ctx = mem.get_context()
    print(f"  {strat:10s}: {len(ctx)} messages in context")
    if strat == "summarize" and mem.summary:
        print(f"              Summary: {mem.summary[:80]}...")
    print()


### Step 4 · Production Chatbot — Everything Together

Reformulation + retrieval + memory + source tracking + streaming-ready.

**`04_production_chatbot.py`** · _Block 4 of 4_

PRODUCTION CONVERSATIONAL RAG CHATBOT


In [ ]:
# PRODUCTION CONVERSATIONAL RAG CHATBOT
import google.generativeai as genai
import numpy as np, os, time

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class ProductionChatbot:
    """Production-ready: reformulate + retrieve + generate + memory + sources."""
    def __init__(self, docs, system_prompt=""):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.system = system_prompt or "You are a helpful Netsetos support assistant. Answer from context only."
        self.history = []
        self.chunks, self.embs = [], []
        for d in docs:
            e = genai.embed_content(model="models/text-embedding-004",content=d["text"])["embedding"]
            self.chunks.append(d); self.embs.append(np.array(e))
        self.stats = {"turns":0, "reformulated":0, "avg_latency":0}

    def chat(self, msg):
        start = time.time()
        # Reformulate
        if self.history:
            h = "\n".join(f"{m['role']}: {m['text'][:60]}" for m in self.history[-6:])
            q = self.model.generate_content(f"Rewrite as standalone. Resolve pronouns.\nHistory:\n{h}\nFollow-up: {msg}\nStandalone:").text.strip()
            self.stats["reformulated"] += 1
        else: q = msg
        # Retrieve
        qe = np.array(genai.embed_content(model="models/text-embedding-004",content=q)["embedding"])
        scores = sorted(enumerate(np.dot(qe,e)/(np.linalg.norm(qe)*np.linalg.norm(e)) for e in self.embs), key=lambda x:-x[1])[:3]
        ctx = "\n".join(self.chunks[i]["text"] for i,_ in scores)
        sources = [self.chunks[i].get("source","?") for i,_ in scores]
        # Generate
        hist = "\n".join(f"{m['role']}: {m['text'][:60]}" for m in self.history[-4:])
        ans = self.model.generate_content(
            f"{self.system}\n\nContext:\n{ctx}\n\nConversation:\n{hist}\nUser: {msg}\nAssistant:").text.strip()
        # Memory
        self.history.append({"role":"user","text":msg})
        self.history.append({"role":"assistant","text":ans})
        self.stats["turns"] += 1
        latency = (time.time()-start)*1000
        return {"answer":ans, "query":q, "sources":sources, "latency_ms":latency}

# ── Full demo conversation ──
docs = [
    {"text":"Netsetos offers GenAI, Data Science, Cloud, Full-Stack courses.","source":"about"},
    {"text":"GenAI course: 14999 rupees. Lifetime access, Discord, live sessions, 5000 GCP credits.","source":"pricing"},
    {"text":"Refund: full 7 days, 50% 8-30 days, none after 30. Email [email protected].","source":"refund"},
    {"text":"Prerequisites: basic Python, high school math. No ML experience needed.","source":"prereqs"},
    {"text":"Live classes 7 PM IST daily. Recordings in 2 hours. EMI via Razorpay.","source":"schedule"},
]

bot = ProductionChatbot(docs)
print("Production Chatbot:\n")
for msg in ["Hi! What GenAI courses do you have?","How much is it?","Can I pay monthly?",
            "What if I want a refund?","How do I request that?","Thanks!"]:
    r = bot.chat(msg)
    print(f"  You: {msg}")
    print(f"  Bot: {r['answer'][:100]}")
    print(f"  [{r['latency_ms']:.0f}ms | query: {r['query'][:50]}]\n")
print(f"Stats: {bot.stats}")


> **What just happened?** LangChain's conversational RAG evolved through three phases. LangGraph is now the recommended approach because it saves entire graph state (not just messages) via checkpointing, enabling fault recovery and human-in-the-loop. RunnableWithMessageHistory is simpler but lacks automatic context window management. Key difference: LangGraph checkpoints at every node boundary; RunnableWithMessageHistory only manages history before/after chain execution.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
